If you're using IDE like VS Code, please install the **Jupyter** extension first.

After you run the **docker-compose up -d** command in your terminal

In [ ]:
docker-compose up -d

 use this command to grab the Jupyter access token: **docker logs jupyter_analytics**

In [ ]:
docker logs jupyter_analytics

You're looking for a link that looks exactly like this:

http://127.0.0.1:8888/lab?token=XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX

Copy that token link. Then, go to your notebook and click **Select Kernel** in the top-right corner.

Go to **Select Another Kernel...** > **Existing Jupyter Server...** and paste your token in the input box.

Hit Enter, then Enter again... and you're in!

🚨 AND DO **NOT** FORGET to run the **First cell** before you do anything else!

# PHASE 1 & 2 Data Preprocessing

In [1]:
# MUST RUN, OR RUN AFTER YOU ARE REFUSED
# ==========================================
# CELL 1: Spark Session Init
# ==========================================
import os
from pyspark.sql import SparkSession

# STEP 1: Set working directory & HDFS identity
os.environ['HADOOP_USER_NAME'] = 'root'
work_dir = '/home/jovyan/work'
if os.path.exists(work_dir):
    os.chdir(work_dir)
print(f"📁 Working Directory: {os.getcwd()}")
print(f"📂 Files found: {os.listdir('.')}")

# STEP 2: Stop any existing Spark session to free memory
existing = SparkSession.getActiveSession()
if existing:
    print(f"🧹 Stopping existing Spark session: {existing.sparkContext.appName}")
    existing.stop()
    print("✅ Old session stopped — JVM memory released.")
else:
    print("ℹ️ No existing Spark session found.")
    
# STEP 3: Start fresh Spark session
# Initialize Spark session with MongoDB connector JARs
MONGO_JARS = ",".join([
    "/home/jovyan/.ivy2/cache/org.mongodb.spark/mongo-spark-connector_2.12/jars/mongo-spark-connector_2.12-10.3.0.jar",
    "/home/jovyan/.ivy2/cache/org.mongodb/mongodb-driver-sync/jars/mongodb-driver-sync-4.8.2.jar",
    "/home/jovyan/.ivy2/cache/org.mongodb/bson/jars/bson-4.8.2.jar",
    "/home/jovyan/.ivy2/cache/org.mongodb/mongodb-driver-core/jars/mongodb-driver-core-4.8.2.jar",  # 4.0.5 → 4.8.2,
])

spark = SparkSession.builder \
    .appName("Ecommerce_Dual_Ingestion_Full") \
    .master("local[*]") \
    .config("spark.jars", MONGO_JARS) \
    .config("spark.driver.memory", "2g") \
    .config("spark.executor.memory", "2g") \
    .config("spark.driver.maxResultSize", "512m") \
    .config("spark.memory.offHeap.enabled", "true") \
    .config("spark.memory.offHeap.size", "1g") \
    .config("spark.sql.shuffle.partitions", "8") \
    .config("spark.memory.fraction", "0.6") \
    .config("spark.memory.storageFraction", "0.3") \
    .config("spark.sql.caseSensitive", "true") \
    .config("spark.hadoop.dfs.client.use.datanode.hostname", "true") \
    .config("spark.hadoop.dfs.datanode.use.datanode.hostname", "true") \
    .getOrCreate()

sc = spark.sparkContext
print(f"✅ Spark Ready!")
print(f"   App Name : {sc.appName}")
print(f"   Version  : {sc.version}")
print(f"   Master   : {sc.master}")
print(f"   Spark UI : http://localhost:4040")

# Check if there is a mongo jar in the container
import glob
all_jars = glob.glob("/home/jovyan/.ivy2/**/*.jar", recursive=True) + \
           glob.glob("/usr/local/spark/jars/**/*.jar", recursive=True)
mongo_jars = [j for j in all_jars if "mongo" in j.lower()]
print("🔍 All mongo jars found:")
for j in mongo_jars:
    print(f"   {j}")

📁 Working Directory: /home/jovyan/work
📂 Files found: ['.git', '.gitignore', 'data', 'datasets.txt', 'data_lake', 'docker-compose.yml', 'hadoop.env', 'meta_review.xlsx', 'proposal', 'requirements.txt', 'venv', '[Phase1&2]fix_meta.py', '[Phase1&2]_PreProcess.ipynb', '[Phase3&4]_Storage&Analytics.ipynb']
ℹ️ No existing Spark session found.
✅ Spark Ready!
   App Name : Ecommerce_Dual_Ingestion_Full
   Version  : 3.5.0
   Master   : local[*]
   Spark UI : http://localhost:4040
🔍 All mongo jars found:


In [2]:
%pip install pymongo

Note: you may need to restart the kernel to use updated packages.


In [ ]:
## Phase 1 & 2
# ==========================================
# CELL 2: Full Ingestion (Parquet & MongoDB)
# ==========================================
import os, shutil
from pymongo import MongoClient

config_file = "datasets.txt"
output_folder = "data_lake"

# --- Path diagnostics ---
print(f"📁 CWD: {os.getcwd()}")
print(f"🔍 Looking for config: {os.path.abspath(config_file)}")
print(f"📂 Files in CWD: {os.listdir('.')}")

# Ensure local data lake folder exists
os.makedirs(output_folder, exist_ok=True)

if not os.path.exists(config_file):
    print(f"\n❌ datasets.txt not found at: {os.path.abspath(config_file)}")
    print("   👉 Make sure datasets.txt is in your mounted project root.")
else:
    with open(config_file, 'r') as f:
        lines = [line.strip() for line in f if line.strip()]

    print(f"\n📋 Found {len(lines)} dataset(s) to process: {lines}")

    for i, line in enumerate(lines):
        dataset_name, table_name = [item.strip() for item in line.split(',')]

        print(f"\n{'='*60}")
        print(f"🔄 [{i+1}/{len(lines)}] Dataset: [{dataset_name}] → table: [{table_name}]")
        print(f"{'='*60}")

        meta_local_in  = os.path.abspath(f"data/meta/meta_{dataset_name}.jsonl")
        rev_local_in   = os.path.abspath(f"data/review/{dataset_name}.jsonl")
        meta_local_out = os.path.abspath(f"{output_folder}/meta_{table_name}.parquet")
        # Note: connector 10.x uses 'mongodb' format and full URI with collection
        mongo_uri = f"mongodb://mongodb:27017/amazon_reviews.{table_name}"

        # ---------------------------------------------------------
        # TASK A: Metadata → Local Data Lake (Parquet, FULL load)
        # ---------------------------------------------------------
        #"""
        df_meta = None
        if os.path.exists(meta_local_in):
            if os.path.exists(meta_local_out):
                shutil.rmtree(meta_local_out)
                print(f"🗑️  [PARQUET] Cleared existing: {meta_local_out}")
            else:
                print(f"ℹ️  [PARQUET] No existing Parquet — fresh write.")
            
            try:
                print(f"\n📦 [META] Reading (full): {meta_local_in}")
                df_meta = spark.read \
                    .option("mode", "DROPMALFORMED") \
                    .json(f"file://{meta_local_in}")

                total_rows = df_meta.count()
                # Each piece is controlled at 20,000 lines to reduce the pressure of a single shuffle
                NUM_PARTITIONS = max(1, total_rows // 20000)
                print(f"   Total rows  : {total_rows:,}")
                print(f"   Partitions  : {NUM_PARTITIONS}")
                print(f"   Columns     : {df_meta.columns}")

                # Cache to disk to avoid re-reading the file
                df_meta = df_meta \
                    .coalesce(NUM_PARTITIONS)  # Coalesce does not trigger shuffle, more memory efficient than repartition  
                
                spark.conf.set("spark.sql.shuffle.partitions", str(NUM_PARTITIONS))
                
                import threading
                import time

                print(f"💾 [META] Writing Parquet in {NUM_PARTITIONS} partition(s) → {meta_local_out}")

                stop_flag = threading.Event()

                def monitor_progress(output_path, total, stop_event):
                    while not stop_event.is_set():
                        if os.path.exists(output_path):
                            done = len([
                                f for f in os.listdir(output_path)
                                if f.endswith(".parquet")
                            ])
                            pct = min(int(done / total * 100), 100)
                            bar = "█" * (pct // 5) + "░" * (20 - pct // 5)
                            print(f"\r   ⏳ [{bar}] {pct}% ({done}/{total} partitions)", end="", flush=True)
                            if done >= total:
                                break
                        time.sleep(1)

                monitor_thread = threading.Thread(
                    target=monitor_progress,
                    args=(meta_local_out, NUM_PARTITIONS, stop_flag)
                )
                monitor_thread.start()

                df_meta.repartition(NUM_PARTITIONS) \
                    .write.mode("overwrite") \
                    .parquet(f"file://{meta_local_out}")

                stop_flag.set()
                monitor_thread.join()
                print(f"\n✅ [META] Done: {table_name} metadata saved.")
            
            except Exception as e:
                print(f"❌ [META] Failed: {str(e)[:200]}")
            finally:
                if df_meta is not None:
                    df_meta.unpersist()
                    spark.catalog.clearCache()
                    print(f"🧹 [META] DataFrame unpersisted & cache cleared.")#"""
        # ---------------------------------------------------------
        # TASK B: Reviews → MongoDB (FULL load)
        # ---------------------------------------------------------
        #"""
        if os.path.exists(rev_local_in):
            # clear MongoDB
            try:
                client = MongoClient("mongodb://mongodb:27017/")
                db = client["amazon_reviews"]
                if table_name in db.list_collection_names():
                    count_before = db[table_name].count_documents({})
                    db[table_name].drop()
                    print(f"🗑️  [MONGO] '{table_name}' dropped — {count_before:,} docs removed.")
                else:
                    print(f"ℹ️  [MONGO] '{table_name}' not found — fresh write.")
                client.close()
            except Exception as e:
                print(f"❌ [MONGO] Pre-clear failed: {e}")

            try:
                import json, os, sys
                from pymongo import MongoClient

                BATCH_SIZE = 50000  # upgrade 5000 to 50000

                client = MongoClient(
                    "mongodb://mongodb:27017/",
                    maxPoolSize=10,
                    socketTimeoutMS=60000,
                    connectTimeoutMS=10000
                )
                collection = client["amazon_reviews"][table_name+'_test']

                file_size = os.path.getsize(rev_local_in)
                print(f"\n📦 [REV] Reading: {rev_local_in}")
                print(f"   File size : {file_size / (1024**3):.2f} GB")

                total_written = 0
                skipped = 0
                batch = []
                bytes_read = 0

                with open(rev_local_in, 'r', encoding='utf-8') as f:
                    for line in f:
                        bytes_read += len(line.encode('utf-8'))
                        line = line.strip()
                        if not line:
                            continue
                        try:
                            batch.append(json.loads(line))
                            if len(batch) >= BATCH_SIZE:
                                collection.insert_many(batch, ordered=False)
                                total_written += len(batch)
                                batch = []

                                pct = min(int(bytes_read / file_size * 100), 99)
                                bar = "█" * (pct // 5) + "░" * (20 - pct // 5)
                                sys.stdout.write(
                                    f"\r   ⏳ [{bar}] {pct}% (~{total_written:,} rows)"
                                )
                                sys.stdout.flush()
                        except json.JSONDecodeError:
                            skipped += 1

                # last batch
                if batch:
                    collection.insert_many(batch, ordered=False)
                    total_written += len(batch)

                client.close()
                print(f"\n✅ [REV] Done!")
                print(f"   Written  : {total_written:,}")
                print(f"   Skipped  : {skipped:,} (malformed JSON)")

            except Exception as e:
                import traceback
                traceback.print_exc()
            finally:
                spark.catalog.clearCache()
                print(f"🧹 [REV] Cache cleared.")
        else:
            print(f"⚠️  [REV] File missing: {rev_local_in}") #"""
    print("\n✨ ALL DUAL-TRACK INGESTION TASKS COMPLETED!")

📁 CWD: /home/jovyan/work
🔍 Looking for config: /home/jovyan/work/datasets.txt
📂 Files in CWD: ['data', 'datasets.txt', 'data_lake', 'docker-compose.yml', 'hadoop.env', 'import_data.ipynb', 'import_data_v1.ipynb', 'mock_data', 'proposal', 'requirements.txt', 'venv']

📋 Found 2 dataset(s) to process: ['Electronics, elec', 'Cell_Phones_and_Accessories, cellp']

🔄 [1/2] Dataset: [Electronics] → table: [elec]
🗑️  [MONGO] 'elec' dropped — 580,000 docs removed.

📦 [REV] Reading & writing line by line: /home/jovyan/work/data/review/Electronics.jsonl
   ⏳ [█████████████████████████████-] 100.00% | 43,885,000 rows
✅ [REV] Done: 43,886,944 rows stored in MongoDB 'elec'.

🔄 [2/2] Dataset: [Cell_Phones_and_Accessories] → table: [cellp]
ℹ️  [MONGO] 'cellp' not found — fresh write.

📦 [REV] Reading & writing line by line: /home/jovyan/work/data/review/Cell_Phones_and_Accessories.jsonl
   ⏳ [█████████████████████████████-]  99.99% | 20,810,000 rows
✅ [REV] Done: 20,812,945 rows stored in MongoDB 'cell

In [ ]:
## Phase 1 & 2
# ==========================================
# CELL 3: Verify MongoDB & Parquet Data Lake
# ==========================================

import os
from pymongo import MongoClient

print("=" * 60)
print("📊 DATA VERIFICATION REPORT")
print("=" * 60)

# Read datasets.txt to obtain the names of all tables
with open("datasets.txt", 'r') as f:
    lines = [line.strip() for line in f if line.strip()]

# ---------------------------------------------------------
# 1. MongoDB Verification
# ---------------------------------------------------------
print("\n🍃 MongoDB Status:")
print("-" * 40)
try:
    client = MongoClient("mongodb://mongodb:27017/")
    db = client["amazon_reviews"]
    existing_collections = db.list_collection_names()
    
    for line in lines:
        dataset_name, table_name = [item.strip() for item in line.split(',')]
        if table_name in existing_collections:
            count = db[table_name].count_documents({})
            sample = db[table_name].find_one()
            fields = list(sample.keys()) if sample else []
            fields = [f for f in fields if f != "_id"]
            print(f"  ✅ Collection : {table_name}")
            print(f"     Documents  : {count:,}")
            print(f"     Fields     : {fields}")
        else:
            print(f"  ❌ Collection '{table_name}' not found in MongoDB.")
    client.close()
except Exception as e:
    print(f"  ❌ MongoDB connection failed: {e}")

# ---------------------------------------------------------
# 2. Parquet Data Lake Verification
# ---------------------------------------------------------
print("\n🗄️  Parquet Data Lake Status:")
print("-" * 40)
output_folder = "data_lake"

for line in lines:
    dataset_name, table_name = [item.strip() for item in line.split(',')]
    parquet_path = os.path.abspath(f"{output_folder}/meta_{table_name}.parquet")
    
    if os.path.exists(parquet_path):
        # Count parquet files and calculate total size
        files = [f for f in os.listdir(parquet_path) if f.endswith(".parquet")]
        total_size = sum(
            os.path.getsize(os.path.join(parquet_path, f))
            for f in files
        )
        size_mb = total_size / (1024 * 1024)
        
        # Verify row count and schema using Spark
        df_check = spark.read.parquet(f"file://{parquet_path}")
        row_count = df_check.count()
        print(f"  ✅ Table      : meta_{table_name}")
        print(f"     Rows       : {row_count:,}")
        print(f"     Partitions : {len(files)} files")
        print(f"     Size       : {size_mb:.1f} MB")
        print(f"     Columns    : {df_check.columns}")
        df_check.unpersist()
    else:
        print(f"  ❌ Parquet 'meta_{table_name}' not found in data_lake/.")

📊 DATA VERIFICATION REPORT

🍃 MongoDB Status:
----------------------------------------
  ✅ Collection : elec
     Documents  : 43,886,944
     Fields     : ['rating', 'title', 'text', 'images', 'asin', 'parent_asin', 'user_id', 'timestamp', 'helpful_vote', 'verified_purchase']
  ✅ Collection : cellp
     Documents  : 20,812,945
     Fields     : ['rating', 'title', 'text', 'images', 'asin', 'parent_asin', 'user_id', 'timestamp', 'helpful_vote', 'verified_purchase']

🗄️  Parquet Data Lake Status:
----------------------------------------
  ✅ Table      : meta_elec
     Rows       : 1,610,012
     Partitions : 80 files
     Size       : 2501.7 MB
     Columns    : ['author', 'average_rating', 'bought_together', 'categories', 'description', 'details', 'features', 'images', 'main_category', 'parent_asin', 'price', 'rating_number', 'store', 'subtitle', 'title', 'videos']
  ✅ Table      : meta_cellp
     Rows       : 1,288,490
     Partitions : 64 files
     Size       : 1248.7 MB
     Column

In [ ]:
## Phase 1 & 2
# FULL MODE
# ==========================================
# CELL 4&5: Show Data From MongoDB & Parquet Data Lake
# ==========================================

# ---------------------------------------------------------
# 1. MongoDB Data Preview
# ---------------------------------------------------------
from pymongo import MongoClient
import pandas as pd

client = MongoClient("mongodb://mongodb:27017/")
database = "amazon_reviews"
db = client[database]
print("\n🗄️ MongoDB Data Preview:\n")
print(f"DataBase: {database}".center(60,'-'))

with open("datasets.txt", 'r') as f:
    lines = [line.strip() for line in f if line.strip()]

for line in lines:
    dataset_name, table_name = [item.strip() for item in line.split(',')]
    
    print(f"\n{'='*60}")
    print(f"📋 Collection: {table_name}")
    print(f"{'='*60}")
    
    if table_name in db.list_collection_names():
        total = db[table_name].count_documents({})
        print(f"   Total docs : {total:,}")
        
        df = pd.DataFrame(list(db[table_name].find({}, {"_id": 0}).limit(1000000)))
        df.info()
        display(df.head())  # Use display() in Jupyter for better formatting
    else:
        print(f"   ❌ Collection '{table_name}' not found.")

client.close()



🗄️ MongoDB Data Preview:

------------------DataBase: amazon_reviews------------------

📋 Collection: elec
   Total docs : 43,886,944
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000000 entries, 0 to 999999
Data columns (total 10 columns):
 #   Column             Non-Null Count    Dtype  
---  ------             --------------    -----  
 0   rating             1000000 non-null  float64
 1   title              1000000 non-null  object 
 2   text               1000000 non-null  object 
 3   images             1000000 non-null  object 
 4   asin               1000000 non-null  object 
 5   parent_asin        1000000 non-null  object 
 6   user_id            1000000 non-null  object 
 7   timestamp          1000000 non-null  int64  
 8   helpful_vote       1000000 non-null  int64  
 9   verified_purchase  1000000 non-null  bool   
dtypes: bool(1), float64(1), int64(2), object(6)
memory usage: 69.6+ MB


,rating,title,text,images,asin,parent_asin,user_id,timestamp,helpful_vote,verified_purchase
0,3.0,Smells like gasoline! Going back!,First & most offensive: they reek of gasoline ...,[{'small_image_url': 'https://m.media-amazon.c...,B083NRGZMM,B083NRGZMM,AFKZENTNBQ7A7V7UXW5JJI6UGRYQ,1658185117948,0,True
1,1.0,Didn’t work at all lenses loose/broken.,These didn’t work. Idk if they were damaged in...,[],B07N69T6TM,B07N69T6TM,AFKZENTNBQ7A7V7UXW5JJI6UGRYQ,1592678549731,0,True
2,5.0,Excellent!,I love these. They even come with a carry case...,[],B01G8JO5F2,B01G8JO5F2,AFKZENTNBQ7A7V7UXW5JJI6UGRYQ,1523093017534,0,True
3,5.0,Great laptop backpack!,I was searching for a sturdy backpack for scho...,[],B001OC5JKY,B001OC5JKY,AGGZ357AO26RQZVRLGU4D4N52DZQ,1290278495000,18,True
4,5.0,Best Headphones in the Fifties price range!,I've bought these headphones three times becau...,[],B013J7WUGC,B07CJYMRWM,AG2L7H23R5LLKDKLBEF2Q3L2MVDA,1676601581238,0,True



📋 Collection: cellp
   Total docs : 20,812,945
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000000 entries, 0 to 999999
Data columns (total 10 columns):
 #   Column             Non-Null Count    Dtype  
---  ------             --------------    -----  
 0   rating             1000000 non-null  float64
 1   title              1000000 non-null  object 
 2   text               1000000 non-null  object 
 3   images             1000000 non-null  object 
 4   asin               1000000 non-null  object 
 5   parent_asin        1000000 non-null  object 
 6   user_id            1000000 non-null  object 
 7   timestamp          1000000 non-null  int64  
 8   helpful_vote       1000000 non-null  int64  
 9   verified_purchase  1000000 non-null  bool   
dtypes: bool(1), float64(1), int64(2), object(6)
memory usage: 69.6+ MB


,rating,title,text,images,asin,parent_asin,user_id,timestamp,helpful_vote,verified_purchase
0,4.0,No white background! It’s clear!,I bought this bc I thought it had the nice whi...,[{'small_image_url': 'https://images-na.ssl-im...,B08L6L3X1S,B08L6L3X1S,AFKZENTNBQ7A7V7UXW5JJI6UGRYQ,1612044451196,0,True
1,5.0,Awesome! Great price! Works well!,Perfect. How pissed am I that I recently paid ...,[],B079BPGF6C,B079BPGF6C,AFKZENTNBQ7A7V7UXW5JJI6UGRYQ,1534443517349,2,True
2,5.0,Worked but took an hour to install,Overall very happy with the end result. If you...,[{'small_image_url': 'https://m.media-amazon.c...,B088DR7Z5B,B0BBGGC8F2,AGCI7FAH4GL5FI65HYLKWTMFZ2CQ,1629235304798,3,True
3,4.0,Decent,Lasted about 9 months then the lock button bro...,[{'small_image_url': 'https://images-na.ssl-im...,B07XRDHDNQ,B07XRDHDNQ,AGCI7FAH4GL5FI65HYLKWTMFZ2CQ,1590470082910,0,True
4,5.0,LOVE IT!,LOVE THIS CASE! Works better than my expensive...,[],B00A8ZDL9Y,B00A8ZDL9Y,AGCI7FAH4GL5FI65HYLKWTMFZ2CQ,1408994588000,0,True


In [ ]:
## Phase 1 & 2
# ---------------------------------------------------------
# 2. Parquet Data Lake Preview
# ---------------------------------------------------------
print("\n🗄️  Parquet Data Lake Preview:")

with open("datasets.txt", 'r') as f:
    lines = [line.strip() for line in f if line.strip()]

for line in lines:
    dataset_name, table_name = [item.strip() for item in line.split(',')]
    parquet_path = os.path.abspath(f"data_lake/meta_{table_name}.parquet")
    
    print(f"\n{'='*60}")
    print(f"📋 Table: meta_{table_name}")
    print(f"{'='*60}")
    
    if os.path.exists(parquet_path):
        df_meta = spark.read.parquet(f"file://{parquet_path}")
        total = df_meta.count()
        print(f"   Total rows : {total:,}")
        slice = df_meta.limit(5).toPandas()
        slice.info()
        display(slice)
        
        df_meta.unpersist()
    else:
        print(f"   ❌ Parquet 'meta_{table_name}' not found.")


🗄️  Parquet Data Lake Preview:

📋 Table: meta_elec
   Total rows : 1,610,012
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5 entries, 0 to 4
Data columns (total 16 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   author           0 non-null      object 
 1   average_rating   5 non-null      float64
 2   bought_together  0 non-null      object 
 3   categories       5 non-null      object 
 4   description      5 non-null      object 
 5   details          5 non-null      object 
 6   features         5 non-null      object 
 7   images           5 non-null      object 
 8   main_category    5 non-null      object 
 9   parent_asin      5 non-null      object 
 10  price            3 non-null      object 
 11  rating_number    5 non-null      int64  
 12  store            5 non-null      object 
 13  subtitle         0 non-null      object 
 14  title            5 non-null      object 
 15  videos           5 non-null      o

,author,average_rating,bought_together,categories,description,details,features,images,main_category,parent_asin,price,rating_number,store,subtitle,title,videos
0,None,3.9,None,"[Electronics, Wearable Technology, Clips, Arm ...",[],"(None, None, None, None, None, None, None, Non...","[Buckle closure, Only compatible waith Garmin ...",[(https://m.media-amazon.com/images/I/61Iru9sg...,AMAZON FASHION,B09GK26HYM,20.99,13,C2D JOY,None,C2D JOY Classic Nylon Weave Band Compatible wi...,[]
1,None,4.4,None,"[Electronics, Television & Video, Accessories,...",[LC-RCRUS-17 Replace Remote Compatible with Sh...,"(None, None, None, None, None, None, None, Non...",[LC-RCRUS-17 Replace Remote Compatible with Sh...,[(https://m.media-amazon.com/images/I/61AWLmk1...,All Electronics,B0777H5CKW,7.98,334,AULCMEET,None,LC-RCRUS-17 Replace Remote Compatible with Sha...,"[(Replacement Remote For Your Roku Device, htt..."
2,None,3.7,None,"[Electronics, Computers & Accessories, Compute...",[The Razer blade 15 mobile Workstation laptop ...,"(None, None, None, None, None, None, None, Non...",[Zero-Compromise Creative Mobile Workstation: ...,[(https://m.media-amazon.com/images/I/71606BPH...,Computers,B07YQP65K7,None,13,Razer,None,Razer Blade 15 Studio Edition Laptop: Intel Co...,[(Microsoft Surface Book 3 Review: Graphics un...
3,None,3.6,None,"[Electronics, Headphones, Earbuds & Accessorie...","[Slight Weight & Sleek Sounds, At a mere 15g, ...","(None, None, None, None, None, None, None, Non...",[【Comfortable and Lightweight】: An ergonomic d...,"[(None, https://m.media-amazon.com/images/I/41...",Home Audio & Theater,B08LVL31PS,None,60,Juckbo,None,"Wireless Earbuds, Magnetic Bluetooth Headphone...",[]
4,None,4.4,None,"[Electronics, Camera & Photo, Lighting & Studi...",[The RadioPopper Nano is the easiest RadioPopp...,"(None, None, None, None, None, None, None, Non...","[Reliable Basic Triggering, Extremely Easy Set...","[(None, https://m.media-amazon.com/images/I/31...",Camera & Photo,B00GMP8Q2Q,59.5,8,RadioPopper,None,RadioPopper NANO-R Nano Receiver (Black),[]



📋 Table: meta_cellp
   Total rows : 1,288,490
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5 entries, 0 to 4
Data columns (total 16 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   author           0 non-null      object 
 1   average_rating   5 non-null      float64
 2   bought_together  0 non-null      object 
 3   categories       5 non-null      object 
 4   description      5 non-null      object 
 5   details          5 non-null      object 
 6   features         5 non-null      object 
 7   images           5 non-null      object 
 8   main_category    5 non-null      object 
 9   parent_asin      5 non-null      object 
 10  price            1 non-null      object 
 11  rating_number    5 non-null      int64  
 12  store            5 non-null      object 
 13  subtitle         0 non-null      object 
 14  title            5 non-null      object 
 15  videos           5 non-null      object 
dtypes: float64(1), int6

,author,average_rating,bought_together,categories,description,details,features,images,main_category,parent_asin,price,rating_number,store,subtitle,title,videos
0,None,3.6,None,"[Cell Phones & Accessories, Wireless Trade-In ...",[],"(None, None, None, None, None, None, None, Non...",[],"[(None, https://m.media-amazon.com/images/I/51...",Cell Phones & Accessories,B00I2KY5TM,None,18,Nokia,None,Nokia Lumia 928 32GB Unlocked GSM 4G LTE Windo...,[]
1,None,4.2,None,"[Cell Phones & Accessories, Accessories, Smart...",[],"(None, None, None, None, None, None, None, Non...",[&#9733 Provided by MP-MALL to ensure the maxi...,[(https://m.media-amazon.com/images/I/61PfxTkg...,Cell Phones & Accessories,B07G839WQ4,None,202,MP-MALL,None,MP-MALL [6-Pack] Screen Protector for Apple Wa...,[(Apple Watch (38mm/42mm) Liquid Skin Screen P...
2,None,5.0,None,"[Cell Phones & Accessories, Accessories, Maint...",[20pcs/lot Mainboard Water Damage Indicator St...,"(None, None, None, None, None, None, None, Non...","[Good Quality, Brand New, Average Price is ver...","[(None, https://m.media-amazon.com/images/I/41...",Cell Phones & Accessories,B00R4HTYHM,None,1,IP4S,None,20pcs/lot Mainboard Water Damage Indicator Sti...,[]
3,None,4.5,None,"[Cell Phones & Accessories, Cases, Holsters & ...","[BENTOBEN, designed by Italy, all the products...","(None, None, None, None, None, None, None, Non...",[★Compatible Phone Models: This cool protectiv...,[(https://m.media-amazon.com/images/I/81MRIzl-...,Cell Phones & Accessories,B08DFK9LM4,None,161,BENTOBEN,None,"BENTOBEN Compatible with LG Stylo 5 Case, LG S...",[]
4,None,4.4,None,"[Cell Phones & Accessories, Cases, Holsters & ...",[Compatible Model:MOTO E 2020 MOTO E Moto e ...,"(None, None, None, None, None, None, None, Non...",[1. Specially designed for MOTO E 2020/Moto e ...,[(https://m.media-amazon.com/images/I/61jwhYb4...,Cell Phones & Accessories,B089XW86DC,8.98,772,Folmeikat,None,"Moto E 2020, Moto e xt2052dl Phone Case,Folmei...","[(Folmeikat Moto E Phone Case,Motorola e Phone..."


In [7]:
import urllib.request
url = "https://repo1.maven.org/maven2/org/mongodb/mongodb-driver-core/4.8.2/mongodb-driver-core-4.8.2.jar"
out = "/home/jovyan/.ivy2/cache/org.mongodb/mongodb-driver-core/jars/mongodb-driver-core-4.8.2.jar"
urllib.request.urlretrieve(url, out)
print(f"✅ Downloaded: {out}")

✅ Downloaded: /home/jovyan/.ivy2/cache/org.mongodb/mongodb-driver-core/jars/mongodb-driver-core-4.8.2.jar


In [13]:
import socket
ip = socket.gethostbyname("mongodb")
print(f"mongodb IP: {ip}")

mongodb IP: 172.18.0.2


In [ ]:
try:
    import socket
    mongo_ip = socket.gethostbyname("mongodb")
    print(f"🔍 MongoDB IP: {mongo_ip}")

    df_rev_sample = spark.read \
        .format("mongodb") \
        .option("spark.mongodb.read.connection.uri", f"mongodb://{mongo_ip}:27017/") \
        .option("spark.mongodb.read.database", "amazon_reviews") \
        .option("spark.mongodb.read.collection", cfg['mongo_col']) \
        .load() \
        .limit(limit_num)
    df_rev_sample.printSchema()
except Exception as e:
    # Print Full Java error
    print(str(e))

🔍 MongoDB IP: 172.18.0.2
root
 |-- _id: string (nullable = true)
 |-- asin: string (nullable = true)
 |-- helpful_vote: integer (nullable = true)
 |-- images: array (nullable = true)
 |    |-- element: struct (containsNull = true)
 |    |    |-- small_image_url: string (nullable = true)
 |    |    |-- medium_image_url: string (nullable = true)
 |    |    |-- large_image_url: string (nullable = true)
 |    |    |-- attachment_type: string (nullable = true)
 |-- parent_asin: string (nullable = true)
 |-- rating: double (nullable = true)
 |-- text: string (nullable = true)
 |-- timestamp: long (nullable = true)
 |-- title: string (nullable = true)
 |-- user_id: string (nullable = true)
 |-- verified_purchase: boolean (nullable = true)



In [15]:
df_meta = spark.read.parquet("file:///home/jovyan/work/data_lake/meta_elec.parquet")
df_meta.select("title", "store", "main_category").show(20, truncate=50)

+--------------------------------------------------+-------------+-------------------------+
|                                             title|        store|            main_category|
+--------------------------------------------------+-------------+-------------------------+
|C2D JOY Classic Nylon Weave Band Compatible wit...|      C2D JOY|           AMAZON FASHION|
|LC-RCRUS-17 Replace Remote Compatible with Shar...|     AULCMEET|          All Electronics|
|Razer Blade 15 Studio Edition Laptop: Intel Cor...|        Razer|                Computers|
|Wireless Earbuds, Magnetic Bluetooth Headphones...|       Juckbo|     Home Audio & Theater|
|          RadioPopper NANO-R Nano Receiver (Black)|  RadioPopper|           Camera & Photo|
|Ear Pads Replacement Compatible K 240 HD 668B H...|        NBBOX|          All Electronics|
|Case for iPad Mini 6 2021: SUPFIVES iPad Mini 6...|     SUPFIVES|                Computers|
|Lilo Stitch TPU Cellphone Case Unique and Fashi...|  Matthew Kim|Cell

In [11]:
# Newly Added
# ==========================================
from pyspark.sql import functions as F

def price_profile(label, parquet_path):
    df = spark.read.parquet(parquet_path)
    if "price" not in df.columns:
        print(f"\n[{label}] no `price` column. Columns: {df.columns}")
        return

    dfp = df.select(F.col("price").cast("string").alias("price_str"))
    total = dfp.count()

    missing_cond = (
        F.col("price_str").isNull()
        | (F.trim(F.col("price_str")) == F.lit(""))
        | F.lower(F.trim(F.col("price_str"))).isin("null", "none", "nan")
    )
    missing = dfp.filter(missing_cond).count()
    miss_rate = (missing / total * 100.0) if total else 0.0

    print(f"\n[{label}] price missing rate")
    print(f"  total rows    : {total:,}")
    print(f"  missing price : {missing:,} ({miss_rate:.2f}%)")

    vc = (
        dfp.select(
            F.when(missing_cond, F.lit("<MISSING>")).otherwise(F.col("price_str")).alias("price_value")
        )
        .groupBy("price_value")
        .count()
        .orderBy(F.col("count").desc(), F.col("price_value"))
    )
    print("  top 1000 value_counts (including <MISSING>):")
    vc.show(1000, truncate=False)

price_profile("meta_elec.parquet", "file:///home/jovyan/work/data_lake/meta_elec.parquet")
price_profile("meta_cellp.parquet", "file:///home/jovyan/work/data_lake/meta_cellp.parquet")


[meta_elec.parquet] price missing rate
  total rows    : 1,610,012
  missing price : 1,083,247 (67.28%)
  top 1000 value_counts (including <MISSING>):
+-----------+-------+
|price_value|count  |
+-----------+-------+
|<MISSING>  |1083247|
|9.99       |15517  |
|19.99      |11481  |
|8.99       |10287  |
|7.99       |9309   |
|12.99      |9304   |
|14.99      |9154   |
|11.99      |9005   |
|29.99      |7655   |
|6.99       |7653   |
|16.99      |7631   |
|15.99      |7480   |
|10.99      |7113   |
|13.99      |6868   |
|39.99      |6259   |
|17.99      |5099   |
|24.99      |5046   |
|18.99      |4851   |
|5.99       |4757   |
|49.99      |4683   |
|21.99      |3441   |
|59.99      |3295   |
|22.99      |3265   |
|25.99      |3212   |
|23.99      |2982   |
|4.99       |2794   |
|69.99      |2730   |
|20.99      |2652   |
|79.99      |2585   |
|9.95       |2519   |
|34.99      |2483   |
|26.99      |2447   |
|99.99      |2426   |
|27.99      |2301   |
|19.95      |2102   |
|32.99      

In [ ]:
import json
from collections import Counter

def profile_price_jsonl(label, jsonl_path, topn=1000, max_lines=None):
    total = 0
    missing = 0
    type_counts = Counter()
    value_counts = Counter()

    with open(jsonl_path, "r", encoding="utf-8") as f:
        for line in f:
            if max_lines is not None and total >= max_lines:
                break
            line = line.strip()
            if not line:
                continue
            total += 1
            try:
                obj = json.loads(line)
            except Exception:
                missing += 1
                type_counts["<MALFORMED_JSON>"] += 1
                continue

            if "price" not in obj or obj.get("price") is None:
                missing += 1
                type_counts["<NULL>"] += 1
                value_counts["<MISSING>"] += 1
                continue

            v = obj.get("price")
            type_counts[type(v).__name__] += 1
            value_counts[str(v)] += 1

    miss_rate = (missing / total * 100.0) if total else 0.0
    scope = f" (first {max_lines:,} lines)" if max_lines is not None else " (full file)"

    print(f"\n[{label}] JSONL price missing rate{scope}")
    print(f"  total rows    : {total:,}")
    print(f"  missing price : {missing:,} ({miss_rate:.2f}%)")
    print("  type counts:")
    for k, c in type_counts.most_common(20):
        print(f"    {k}: {c:,}")
    print(f"  top {topn} value_counts (including <MISSING>):")
    for v, c in value_counts.most_common(topn):
        print(f"    {v}\t{c}")

with open("datasets.txt", 'r') as f:
    lines = [line.strip() for line in f if line.strip()]

for line in lines:
    profile_price_jsonl(f"meta_{line.split(',')[0].strip()}.jsonl", f"/home/jovyan/work/data/meta/meta_{line.split(',')[0].strip()}.jsonl")



[meta_Electronics.jsonl] JSONL price missing rate (full file)
  total rows    : 1,610,012
  missing price : 1,083,247 (67.28%)
  type counts:
    <NULL>: 1,083,247
    float: 526,449
    str: 316
  top 1000 value_counts (including <MISSING>):
    <MISSING>	1083247
    9.99	15517
    19.99	11481
    8.99	10287
    7.99	9309
    12.99	9304
    14.99	9154
    11.99	9005
    29.99	7655
    6.99	7653
    16.99	7631
    15.99	7480
    10.99	7113
    13.99	6868
    39.99	6259
    17.99	5099
    24.99	5046
    18.99	4851
    5.99	4757
    49.99	4683
    21.99	3441
    59.99	3295
    22.99	3265
    25.99	3212
    23.99	2982
    4.99	2794
    69.99	2730
    20.99	2652
    79.99	2585
    9.95	2519
    34.99	2483
    26.99	2447
    99.99	2426
    27.99	2301
    19.95	2102
    32.99	2049
    89.99	2014
    35.99	1946
    28.99	1927
    14.95	1635
    12.95	1599
    36.99	1498
    7.95	1470
    29.95	1433
    31.99	1429
    129.99	1418
    33.99	1350
    24.95	1341
    11.95	1335
    149.99	1331
  

In [14]:
# check missing price in jsonl file
from pyspark.sql import functions as F

def analyze_missing_reason(label, jsonl_path):
    df = spark.read.option("mode", "DROPMALFORMED").json(f"file://{jsonl_path}")

    cols = [c for c in ["parent_asin", "asin", "store", "main_category", "price"] if c in df.columns]
    df = df.select(cols)

    if "price" not in df.columns:
        print(f"\n[{label}] no price column in JSON")
        return

    price_str = F.col("price").cast("string")
    missing = price_str.isNull() | (F.trim(price_str) == F.lit("")) | F.lower(F.trim(price_str)).isin("null", "none", "nan")
    df = df.withColumn("price_missing", missing)

    total = df.count()
    miss = df.filter(F.col("price_missing")).count()
    print(f"\n[{label}] total={total:,}, missing={miss:,} ({(miss/total*100.0) if total else 0.0:.2f}%)")

    if "store" in df.columns:
        print("\nMissing vs store (top 30 counts):")
        df.filter(F.col("price_missing")) \
          .groupBy("store").count() \
          .orderBy(F.col("count").desc()) \
          .show(30, truncate=False)

        print("\nNon-missing vs store (top 30 counts):")
        df.filter(~F.col("price_missing")) \
          .groupBy("store").count() \
          .orderBy(F.col("count").desc()) \
          .show(30, truncate=False)

    if "main_category" in df.columns:
        print("\nMissing rate by main_category (top 30 by total):")
        agg = df.groupBy("main_category").agg(
            F.count("*").alias("total"),
            F.sum(F.col("price_missing").cast("int")).alias("missing")
        ).withColumn("missing_rate", F.col("missing") / F.col("total"))
        agg.orderBy(F.col("total").desc()).show(30, truncate=False)

    print("\nSample rows with missing price (20):")
    df.filter(F.col("price_missing")).limit(20).show(20, truncate=60)

with open("datasets.txt", 'r') as f:
    lines = [line.strip() for line in f if line.strip()]
    
for line in lines:
    analyze_missing_reason(f"meta_{line.split(',')[0].strip()}.jsonl", f"/home/jovyan/work/data/meta/meta_{line.split(',')[0].strip()}.jsonl")



[meta_Electronics.jsonl] total=1,610,012, missing=1,083,247 (67.28%)

Missing vs store (top 30 counts):
+--------------+-----+
|store         |count|
+--------------+-----+
|Amazon Renewed|19543|
|HP            |10601|
|Sony          |10317|
|NULL          |8916 |
|SAMSUNG       |8525 |
|Generic       |8429 |
|Lenovo        |5876 |
|Neewer        |5805 |
|ASUS          |4567 |
|Fintie        |4518 |
|Dell          |4031 |
|Canon         |3555 |
|Panasonic     |2900 |
|Dteck         |2724 |
|Belkin        |2599 |
|Apple         |2481 |
|Garmin        |2365 |
|LG            |2339 |
|MOSISO        |2247 |
|acer          |2223 |
|Toshiba       |2205 |
|uxcell        |2170 |
|Unknown       |2119 |
|INFILAND      |2062 |
|HDE           |2012 |
|Nikon         |1927 |
|JVC           |1863 |
|Fujifilm      |1847 |
|SquareTrade   |1830 |
|Pyle          |1657 |
+--------------+-----+
only showing top 30 rows


Non-missing vs store (top 30 counts):
+---------------+-----+
|store          |count|


In [16]:
# noticed that the missing rate of price is around 70% high
from pyspark.sql import functions as F

with open("datasets.txt", 'r') as f:
    lines = [line.strip() for line in f if line.strip()]
    
for line in lines:
   df = spark.read.parquet(f"file:///home/jovyan/work/data_lake/meta_{line.split(',')[1].strip()}.parquet")  # 或 meta_cellp
   dfp = df.select(F.col("price").cast("string").alias("price_str"))

   missing_type = (
      F.when(F.col("price_str").isNull(), F.lit("NULL"))
      .when(F.trim(F.col("price_str")) == F.lit(""), F.lit("EMPTY_OR_WHITESPACE"))
      .when(F.lower(F.trim(F.col("price_str"))).isin("null", "none", "nan"), F.lit("STRING_NULL_LIKE"))
      .otherwise(F.lit("NOT_MISSING"))
   )

   dfp.withColumn("missing_type", missing_type) \
      .groupBy("missing_type").count() \
      .orderBy(F.col("count").desc()) \
      .show(truncate=False)

   dfp.withColumn("missing_type", missing_type) \
      .filter(F.col("missing_type") != "NOT_MISSING") \
      .select("missing_type", "price_str", F.length("price_str").alias("len"), F.lower(F.trim("price_str")).alias("lower_trim")) \
      .show(50, truncate=False)

['Electronics, elec', 'Cell_Phones_and_Accessories, cellp']


AnalysisException: [PATH_NOT_FOUND] Path does not exist: file:/home/jovyan/work/data_lake/meta_ elec.parquet.

In [17]:
from pyspark.sql.functions import col

df_meta.groupBy("main_category") \
    .count() \
    .orderBy(col("count").desc()) \
    .show(50, truncate=False)



+----------------------------+------+
|main_category               |count |
+----------------------------+------+
|Computers                   |418868|
|All Electronics             |376435|
|Camera & Photo              |223690|
|Cell Phones & Accessories   |138237|
|Home Audio & Theater        |106516|
|NULL                        |106334|
|Industrial & Scientific     |50555 |
|Car Electronics             |27332 |
|Tools & Home Improvement    |24042 |
|Amazon Home                 |21299 |
|Office Products             |21250 |
|AMAZON FASHION              |18391 |
|Sports & Outdoors           |14581 |
|Automotive                  |13956 |
|GPS & Navigation            |9339  |
|Amazon Devices              |8484  |
|Portable Audio & Accessories|8050  |
|Musical Instruments         |6668  |
|Toys & Games                |3944  |
|Health & Personal Care      |3073  |
|All Beauty                  |1848  |
|Arts, Crafts & Sewing       |1496  |
|Apple Products              |1078  |
|Video Games

In [18]:
df_meta.groupBy("store") \
    .count() \
    .orderBy(col("count").desc()) \
    .show(100, truncate=False)

+------------------+-----+
|store             |count|
+------------------+-----+
|Amazon Renewed    |30576|
|HP                |17129|
|Sony              |14160|
|Generic           |13499|
|SAMSUNG           |11503|
|NULL              |9520 |
|Lenovo            |9049 |
|Dell              |7042 |
|ASUS              |6870 |
|Neewer            |6755 |
|Fintie            |5370 |
|Canon             |5125 |
|UPBRIGHT          |4574 |
|uxcell            |4386 |
|Panasonic         |4039 |
|Kastar            |3679 |
|Garmin            |3608 |
|Belkin            |3343 |
|Synergy Digital   |3148 |
|Nikon             |3116 |
|HQRP              |3074 |
|LG                |3031 |
|Monoprice         |2974 |
|MOSISO            |2890 |
|Dteck             |2848 |
|Apple             |2847 |
|MIGHTY SKINS      |2784 |
|Intel             |2752 |
|Toshiba           |2735 |
|acer              |2712 |
|Pyle              |2681 |
|Unknown           |2647 |
|Fujifilm          |2632 |
|Logitech          |2540 |
|

In [4]:
%pip install pyarrow

Note: you may need to restart the kernel to use updated packages.


In [ ]:
# ==========================================
# Brand Whitelist Filter → Metadata Clean → Review Alignment (Chunked)
# ==========================================

import os
import time
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
from pymongo import MongoClient
from pyspark.sql import functions as F
from pyspark.sql.types import FloatType, StructType, StructField, StringType, IntegerType, BooleanType, LongType

output_dir = os.path.abspath("data_lake")

# ─────────────────────────────────────────
# CONFIG: Brand Whitelist
# ─────────────────────────────────────────
BRAND_WHITELIST = [
    # Smartphones / Tablets
    "apple", "samsung", "google", "sony", "oneplus", "xiaomi", "oppo", "huawei",
    "motorola", "nokia", "realme", "vivo", "zte", "blackberry", "honor",
    # Laptops / PCs
    "dell", "hp", "lenovo", "asus", "acer", "microsoft", "razer", "msi",
    "toshiba", "gigabyte", "lg", "alienware", "system76",
    # Audio
    "bose", "jbl", "sennheiser", "jabra", "anker", "soundcore",
    "beats", "shure", "audio-technica", "skullcandy",
    "plantronics", "beyerdynamic", "marshall",
    # Gaming Peripherals
    "logitech", "corsair", "steelseries", "hyperx", "roccat",
    "zowie", "glorious", "ducky",
    # Cameras
    "canon", "nikon", "fujifilm", "gopro",
    "panasonic", "olympus", "leica", "dji", "insta360", "sigma", "pentax",
    # Displays / TVs
    "tcl", "hisense", "vizio", "philips",
    "viewsonic", "benq", "aoc",
]
BRAND_SET_LOWER = set(b.lower() for b in BRAND_WHITELIST)

# Meta columns we want
META_COLS = ["parent_asin", "title", "price", "store", "main_category",
             "average_rating", "rating_number", "details"]

# Review schema
REV_SCHEMA = StructType([
    StructField("parent_asin",       StringType(),  True),
    StructField("asin",              StringType(),  True),
    StructField("rating",            FloatType(),   True),
    StructField("title",             StringType(),  True),
    StructField("text",              StringType(),  True),
    StructField("timestamp",         LongType(),    True),
    StructField("helpful_vote",      IntegerType(), True),
    StructField("verified_purchase", BooleanType(), True),
    StructField("user_id",           StringType(),  True),
])
REV_COLS = [f.name for f in REV_SCHEMA.fields]

# ─────────────────────────────────────────
# Dataset loop
# ─────────────────────────────────────────
with open("datasets.txt", 'r') as f:
    lines = [line.strip() for line in f if line.strip()]

for id,line in enumerate(lines):
    print(id,line)
    if id == 0:
        continue
    dataset_name, table_name = [x.strip() for x in line.split(',')]
    print(dataset_name, table_name)
    t0 = time.time()
    print(f"\n{'='*60}")
    print(f"🔄 Processing: [{dataset_name}]")
    print(f"{'='*60}")

    # ── STEP 1: Load metadata Parquet ──────────────────────────
    meta_in = f"file://{os.path.abspath(f'data_lake/meta_{table_name}.parquet')}"
    print(f"\n📦 [META] Loading parquet: {meta_in}")
    df_meta = spark.read.parquet(meta_in)

    # ── STEP 2: Brand whitelist filter ─────────────────────────
    df_meta_filtered = df_meta.filter(
        F.lower(F.col("store")).isin(list(BRAND_SET_LOWER))
    )
    count_before = df_meta.count()
    count_after  = df_meta_filtered.count()
    print(f"   Rows before : {count_before:,}")
    print(f"   Rows after  : {count_after:,}  ({count_after/count_before*100:.1f}% kept)")

    # ── STEP 3: Select columns + cast price to float ───────────
    df_meta_clean = df_meta_filtered.select(
        "parent_asin",
        "title",
        F.col("price").cast(FloatType()).alias("price"),
        "store",
        "main_category",
        F.col("average_rating").cast(FloatType()).alias("average_rating"),
        "rating_number",
        "details",
    )

    # ── STEP 4: Collect valid parent_asin set ──────────────────
    valid_asins = [
        row["parent_asin"]
        for row in df_meta_clean.select("parent_asin").distinct().collect()
    ]
    print(f"   Valid parent_asin: {len(valid_asins):,}")

    # ── STEP 5: Chunked Extraction to Parquet (NO OOM) ─────────
    print(f"\n📦 [REV] Querying MongoDB '{table_name}' with $in ...")
    t_rev = time.time()

    client = MongoClient("mongodb://mongodb:27017/", maxPoolSize=10, socketTimeoutMS=300000)
    collection = client["amazon_reviews"][table_name]

    cursor = collection.find(
        {"parent_asin": {"$in": valid_asins}},
        {"_id": 0, **{col: 1 for col in REV_COLS}}
    ).batch_size(50000)

    # PyArrow File Path (Native OS Path)
    rev_out_path = os.path.join(output_dir, f"review_{dataset_name}.parquet")
    
    batch_data = []
    writer = None
    total_fetched = 0

    # Core change: avoid unbounded appends by flushing every 100k rows
    for doc in cursor:
        batch_data.append({k: doc.get(k) for k in REV_COLS})
        total_fetched += 1
        
        if len(batch_data) >= 100000:
            # transform to Pandas and PyArrow format and write
            df_batch = pd.DataFrame(batch_data)
            table = pa.Table.from_pandas(df_batch)
            
            if writer is None:
                writer = pq.ParquetWriter(rev_out_path, table.schema)
                
            writer.write_table(table)
            print(f"   ⏳ {total_fetched:,} reviews fetched & saved so far...")
            batch_data = [] # 🚀 Clear list to free memory

    # Write the last remaining batch
    if len(batch_data) > 0:
        df_batch = pd.DataFrame(batch_data)
        table = pa.Table.from_pandas(df_batch)
        if writer is None:
            writer = pq.ParquetWriter(rev_out_path, table.schema)
        writer.write_table(table)
        print(f"   ⏳ {total_fetched:,} reviews saved (Final chunk).")

    if writer:
        writer.close()
    client.close()
    print(f"✅ [REV] Successfully extracted and saved {total_fetched:,} reviews in {time.time()-t_rev:.1f}s")

    # ── STEP 6: Save cleaned metadata → Parquet ───────────────
    meta_out = f"file://{output_dir}/meta_{dataset_name}.parquet"
    print(f"\n💾 Saving metadata → meta_{dataset_name}.parquet")
    df_meta_clean.write.mode("overwrite").parquet(meta_out)
    print(f"✅ [META] Saved.")

    # ── STEP 7: REMOVED (Handled in Step 5 natively) ──────────。
    
    # Cleanup
    df_meta.unpersist()
    df_meta_clean.unpersist()
    spark.catalog.clearCache()
    print(f"\n⏱️  [{dataset_name}] Total time: {time.time()-t0:.1f}s")

print("\n✨ ALL DATASETS CLEANED AND SAVED!")
# There are bugs saving metadataparquet files, if fail again, use local fix_meta.py file to fix it.

0 Electronics, elec
1 Cell_Phones_and_Accessories, cellp
Cell_Phones_and_Accessories cellp

🔄 Processing: [Cell_Phones_and_Accessories]

📦 [META] Loading parquet: file:///home/jovyan/work/data_lake/meta_cellp.parquet
   Rows before : 1,288,490
   Rows after  : 17,801  (1.4% kept)
   Valid parent_asin: 17,801

📦 [REV] Querying MongoDB 'cellp' with $in ...
   ⏳ 100,000 reviews fetched & saved so far...
   ⏳ 200,000 reviews fetched & saved so far...
   ⏳ 300,000 reviews fetched & saved so far...
   ⏳ 400,000 reviews fetched & saved so far...
   ⏳ 500,000 reviews fetched & saved so far...
   ⏳ 600,000 reviews fetched & saved so far...
   ⏳ 700,000 reviews fetched & saved so far...
   ⏳ 800,000 reviews fetched & saved so far...
   ⏳ 900,000 reviews fetched & saved so far...
   ⏳ 960,550 reviews saved (Final chunk).
✅ [REV] Successfully extracted and saved 960,550 reviews in 229.2s

💾 Saving metadata → Cell_Phones_and_Accessories_meta_clean.parquet


ERROR:root:Exception while sending command.
Traceback (most recent call last):
  File "/usr/local/spark/python/lib/py4j-0.10.9.7-src.zip/py4j/clientserver.py", line 516, in send_command
    raise Py4JNetworkError("Answer from Java side is empty")
py4j.protocol.Py4JNetworkError: Answer from Java side is empty

During handling of the above exception, another exception occurred:

Traceback (most recent call last):
  File "/usr/local/spark/python/lib/py4j-0.10.9.7-src.zip/py4j/java_gateway.py", line 1038, in send_command
    response = connection.send_command(command)
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/spark/python/lib/py4j-0.10.9.7-src.zip/py4j/clientserver.py", line 539, in send_command
    raise Py4JNetworkError(
py4j.protocol.Py4JNetworkError: Error while sending or receiving


Py4JError: An error occurred while calling o159.parquet

ERROR:root:Exception while sending command.
Traceback (most recent call last):
  File "/usr/local/spark/python/lib/py4j-0.10.9.7-src.zip/py4j/clientserver.py", line 516, in send_command
    raise Py4JNetworkError("Answer from Java side is empty")
py4j.protocol.Py4JNetworkError: Answer from Java side is empty

During handling of the above exception, another exception occurred:

Traceback (most recent call last):
  File "/usr/local/spark/python/lib/py4j-0.10.9.7-src.zip/py4j/java_gateway.py", line 1038, in send_command
    response = connection.send_command(command)
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/spark/python/lib/py4j-0.10.9.7-src.zip/py4j/clientserver.py", line 539, in send_command
    raise Py4JNetworkError(
py4j.protocol.Py4JNetworkError: Error while sending or receiving


In [ ]:
import os
import pandas as pd
import pyarrow.dataset as ds

with open("datasets.txt", 'r') as f:
    lines = [line.strip() for line in f if line.strip()]
# The file path of the data lake to be inspected
files_to_check = [
    f"data_lake/{types}_{line.split(',')[0].strip()}.parquet"
    for line in lines
    for types in ["review", "meta"]
]

print("🔍 Starting final validation of the Data Lake...\n")

for f in files_to_check:
    path = os.path.abspath(f)
    if os.path.exists(path):
        # Calculate the disk usage of the entire folder (or single file)
        size_mb = 0
        if os.path.isfile(path):
            size_mb = os.path.getsize(path) / (1024 * 1024)
        else:
            size_mb = sum(os.path.getsize(os.path.join(dirpath, filename)) 
                          for dirpath, dirnames, filenames in os.walk(path) 
                          for filename in filenames) / (1024 * 1024)
        
        print(f"📂 Target path: {f}")
        print(f"   📏 Disk usage: {size_mb:.2f} MB")
        
        try:
            # Use PyArrow Dataset for fast reading
            dataset = ds.dataset(path, format="parquet")
            
            # Quickly get the accurate total row count (without loading all data into memory)
            actual_rows = dataset.count_rows()
            
            # Only take the first 2 rows as a sample
            df_head = dataset.head(2).to_pandas()
            
            print(f"   🔢 Actual total rows: {actual_rows:,}")
            print(f"   📋 Columns: {df_head.columns.tolist()}")
            print(f"   ✅ Data preview:")
            display(df_head)
            
        except Exception as e:
            print(f"   ❌ Failed to read: {e}")
    else:
        print(f"⚠️ Path not found: {f}")
        
    print("-" * 70)

🔍 Starting final validation of the Data Lake...

📂 Target path: data_lake/review_Electronics_clean.parquet
   📏 Disk usage: 1468.53 MB
   🔢 Actual total rows: 5,984,931
   📋 Columns: ['parent_asin', 'asin', 'rating', 'title', 'text', 'timestamp', 'helpful_vote', 'verified_purchase', 'user_id']
   ✅ Data preview:


,parent_asin,asin,rating,title,text,timestamp,helpful_vote,verified_purchase,user_id
0,0246003235,0246003235,5.0,SD card,It works.,1490823998000,0,True,AHKEPQU5OXUSN2573LYB2VHN6H5Q
1,0246003235,0246003235,5.0,Works!,Perfect for my needs.,1496318540000,0,True,AFAPRKGC56RBEZREID7RHEZ6OD7Q


----------------------------------------------------------------------
📂 Target path: data_lake/meta_Electronics_clean.parquet
   📏 Disk usage: 141.50 MB
   🔢 Actual total rows: 128,428
   📋 Columns: ['parent_asin', 'title', 'price', 'store', 'main_category', 'average_rating', 'rating_number', 'details']
   ✅ Data preview:


,parent_asin,title,price,store,main_category,average_rating,rating_number,details
0,B000NDDL5U,Shure PA658 Airline Adapter for Shure Earphones,None,Shure,Home Audio & Theater,3.7,6,"{'63 99': None, '74 89': None, '89 89': None, ..."
1,B0000E1FDJ,Sony External USB Floppy Drive (MPF88E/UA/181),None,Sony,All Electronics,3.9,58,"{'63 99': None, '74 89': None, '89 89': None, ..."


----------------------------------------------------------------------
📂 Target path: data_lake/review_Cell_Phones_and_Accessories_clean.parquet
   📏 Disk usage: 190.24 MB
   🔢 Actual total rows: 960,550
   📋 Columns: ['parent_asin', 'asin', 'rating', 'title', 'text', 'timestamp', 'helpful_vote', 'verified_purchase', 'user_id']
   ✅ Data preview:


,parent_asin,asin,rating,title,text,timestamp,helpful_vote,verified_purchase,user_id
0,1466736038,1466736038,5.0,Five Stars,awesome phone! would but this again!,1458082064000,0,True,AGKFD2QITYYIN5UTP4ONM35XLWAQ
1,1466736038,1466736038,5.0,SGH-I747 2-1-15,i enjoyed this a few years ago and don't mind ...,1422811603000,0,True,AHMCFS7Q5ZCT56EKERCFWLUGTNFA


----------------------------------------------------------------------
📂 Target path: data_lake/meta_Cell_Phones_and_Accessories_clean.parquet
   📏 Disk usage: 21.69 MB
   🔢 Actual total rows: 17,801
   📋 Columns: ['parent_asin', 'title', 'price', 'store', 'main_category', 'average_rating', 'rating_number', 'details']
   ✅ Data preview:


,parent_asin,title,price,store,main_category,average_rating,rating_number,details
0,B00BWLR8DS,HTC One 32GB Unlocked GSM Android Cell Phone w...,None,Nokia,All Electronics,4.5,7,"{'AC Adapter Current': None, 'Accessory Connec..."
1,B00524T47Y,Lcd Screen Backlight Replacement for Psp 2000 ...,None,Sony,Cell Phones & Accessories,4.3,14,"{'AC Adapter Current': None, 'Accessory Connec..."


----------------------------------------------------------------------


In [ ]:
# Release the memory of MongoDB